# Simulation of an exchange kinetics between 3 states  

$\mathcal{S}_C \rightleftharpoons  \mathcal{S}_A \rightleftharpoons \mathcal{S}_B$

## Introduction

### Objective of the exercices
1. Get accustomed the matricial notations used in Markov State Models theory.
2. Recognize the exponential changes of the evolution of the system with different timescales.
4. Investigate the information contained in eigenvalues and eigenvectors of the propagator.


### Context

We consider again a solution of molecules in exchange between 3 states $S_A$, $S_B$ and  $S_C$, 
with concentration $[C_A](t)$, $[C_B](t)$, $[C_C](t)$  as a function of time.

The total concentration is noted $[C](t) = [C_A](t) + [C_B](t)+ [C_C](t) $ is constant.

The kinetic constant of the reaction from $\mathcal{S}_A$ to $\mathcal{S}_B$ is noted $k_{AB}$. 
The one from $B$ to $A$ is noted  $k_{BA}$. Siilar for the exchange within A and B

We shall simulate  an experiment with a starting compositions $[C_A]_i$, $[C_B]_i$, $[C_C]_i$, and predict the time evolution.

This time the analytical calculation is not easy. We shall 
calculate numerically and play with the initial conditions.

 

### Notations
- $t$ the time.
- $\tau$ a small time interval (also named timelag or timestep).
- $S_i$  : states $A$, $B$ and $C$, with concentration $[C_i](t)$.
- The kinetic constant of the reaction from $\mathcal{S}_i$ to $\mathcal{S}_j$ is noted $k_{ij}$.
- $K^{eq}$ is the equilibrium constant of the equilibrium i.e. $K_{AB}^{eq} = [C_B]/[C_A] $ at equilibrium.
- $p_i(t)$ : probabilities to be in state $i$, at the time $t$.
- $p_i^{eq}$ :  the corresponding probabilities at equilibrium.

In [ ]:
# import all necessary packages
import sys
from sympy import *
import mpmath
import math
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib import cm
import string

sys.modules['sympy.mpmath'] = mpmath

In [ ]:
# define the different symbolic calculation variables

pAi = symbols('pai') # initial concentration in A
pBi = symbols('pbi') # initial concentration in B
pCi = symbols('pci') # initial concentration in C


kAB = symbols('k_AB') # kinetic constant from A to B
kBA = symbols('k_BA') # kinetic constant from B to A
kAC = symbols('k_AC') # kinetic constant from A to C
kCA = symbols('k_CA') # kinetic constant from C to A

KeqAB = kAB/kBA # equilibrium constant pB/pA
KeqAC = kAC/kCA # equilibrium constant pC/pA


t = symbols('t') # time 
tau = symbols('tau') # small time step 


In [ ]:
# write the initial probability as a vector 
#  a list of elements is considered to be a column vector.
pi = Matrix([pAi,pBi,pCi])
print(pi)

# the first index of the matrix is the length of the column = the number of rows
# the second index is the number of column
print(shape(pi))

## Differential equations 



**QUESTIONS**

1. Make a scheme of the exchange with the kinetic constants. 
2. Write how $[C_A](t)$ depends on $C$ and $p_A(t)$. Same question for $B$ and $C$.
Write the three time-differential equations for $p_i(t)$, Eq.(i).
3. We define a vector 
$ p =
\begin{pmatrix} 
p_A\\
p_B\\
p_C
\end{pmatrix}
$. 
Write the differential equation for $p$ with the form 
$
\frac{dp}{dt} = \hat{k} p
$
where $\hat{k}$ is a $3 \times 3$ matrix that is independant of $t$. This equation is noted Eq.(K).
4. Complete the following cell to  define k and let it run without error.

In [ ]:

alphabet = list(string.ascii_uppercase)

# Matrix that describes the time derivative of the probabilities
# the matrix is given  rows by rows
# the row i  provides the list of coefficients for the  time derivative  of the state i 
k = Matrix([[-(kAB+kAC), kBA, kCA], [kAB, -kBA, 0], [kAC, 0, -kCA]])
#k = TOCOMPLETE

k_dimension = shape(k)[0]

print(f'Matrix k is = {k}')
print(f'shape of the k matrix = {shape(k)}')
print(f'number of rows    = {shape(k)[0]}')
print(f'number of columns = {shape(k)[1]}')
for row in np.arange(shape(k)[0]) :
    print(f'for compound {alphabet[row]} : coef. kinetics = {list(k.row(row))}')



Let us think again about the eigenvalues and eigenvectors of $\hat{k}$. 

**QUESTIONS**

1. What does the eigenvector with eigenvalue 0 bring for information ? 
3. in the following cell, you will add some code to let `sympy` do the calculation for you.


In [ ]:

# for 
#print(k.col(0))

# describe the eigenvector of this matrix
print(f'=> eigenvectors of k ')
eigen = k.eigenvects()
for e in eigen :
    eigenvalue, multiplicity, vector = e
    print(f'eigenvalue = {eigenvalue.simplify()}, multiplicity = {multiplicity}, vector = {vector}')



# investigate the eigenvalues of the propagator

print(f"Find the eigenvalue 0...")
for e in eigen :
    eigenvalue, multiplicity, vector = e   
    if eigenvalue == 0 :  #? NOTE: previous version (which failed) was eigenvalue == 0.0
        print(f"Eigenvalues = 0  found! \nThis eigenvector describes the equilibrium distribution ")
        vector0 = vector[0]
        print(f'eigenvector = {vector0}')

        sum = 0.0
        for value in vector0:
            sum=sum+value
        vector_normalized = vector0/sum
        expr_pA_eq = vector_normalized[0].simplify()
        expr_pB_eq = vector_normalized[1].simplify()
        expr_pC_eq = vector_normalized[2].simplify()

        print(f"equilibrium distribution :")
        print(f'pA_eq = {expr_pA_eq}')
        print(f'pB_eq = {expr_pB_eq}')
        print(f'pC_eq = {expr_pC_eq}')


pieq = Matrix([expr_pA_eq,expr_pB_eq,expr_pC_eq])
#print(pieq)
#print(f'=> Action of matrix k onto vector p')
#print(k*pi)


## Numerical application

**QUESTIONS**

1. in the following cell,  define some constants where the equilibrium $A/B$ is ten times more rapid than the equilibrium $A/C$, and see the evolution of the concentration. Do you recognize the two timeranges ?

In [ ]:

# make a numerical application a see how the proabilities evolve in time
subs = {kAB : 8.0, kBA : 2.0, kAC : 0.40, kCA : 0.2, pAi : 1.0 , pBi :0.0, pCi :0.0, tau: 0.05}
print(f'Dictionnary of the parameter values = {subs}')


In [ ]:
# same kinetics, change for PROPAGATOR formalism
# P is the propagator
# It describes the probability after a small time tau depending on the probability at time t
P = eye(k_dimension) + tau * k 
P_dimension = shape(P)[0]

print(f"Description of the propagator for a timestep {tau}")
for row in np.arange(P_dimension) :
    print(f'for compound {alphabet[row]} : coef. propagator = {list(P.row(row))}')

# investigate the eigenvalues of the propagator
print(f"Eigenvalues of the propagator")
eigen = P.eigenvects()
for e in eigen :
    eigenvalue, multiplicity, vector = e
    print(f'=> eigenvalue = {eigenvalue}, multiplicity = {multiplicity}, vector = {vector}')

print(f"Find the eigenvalue 1...")
for e in eigen :
    eigenvalue, multiplicity, vector = e   
    if eigenvalue == 1 :  #? NOTE: previous version (which failed) was eigenvalue == 1.0
        print(f"Eigenvalues = 1  ! \n This eigenvector describes the equilibrium distribution ")
        vector0 = vector[0]
        print(f'eigenvector = {vector0}')

        sum = 0.0
        for value in vector0:
            sum=sum+value
        vector_normalized = vector0/sum
        exprA_eq = vector_normalized[0].simplify()
        exprB_eq = vector_normalized[1].simplify()
        exprC_eq = vector_normalized[2].simplify()

        print(f"equilibrium distribution :")
        print(f'pA_eq = {exprA_eq}')
        print(f'pB_eq = {exprB_eq}')
        print(f'pC_eq = {exprC_eq}')


In [ ]:
# Calculate again the time evolution using the propagator
# using the numerical 
P_num = P.evalf(5,subs=subs)

print(f'Numerical value of the propagator P for a timestep {tau.evalf(2,subs=subs)}')
for row in np.arange(P_dimension) :
    print(f'for compound {alphabet[row]} : P_{row} = {list(P_num.row(row))}')

In [ ]:
# calculate the time evolution of the concentration
# p(t+tau) = P(tau) * p(t)

# define reasonable time range, avoid to divide by 0
tmax = float((100/(kAB+kBA)).evalf(subs=subs))
tstep = float(tau.evalf(subs=subs))
t_list= np.arange(0.001,tmax,tstep)

# initialize the vector p(t) at time 0 and put in a list that will evolve in time
# the first index of the matrix is the length of the column = the number of rows
# the second index is the number of column
pi_num = Matrix([pAi,pBi,pCi]).evalf(5,subs=subs)
pt_tau_list = [ pi_num ]

# create an array to store the values in a convenient way for plotting
# initialize it at zero values from pt_tau_list
shape = (P_dimension,len(t_list))
pt_array = np.zeros(shape=shape, dtype=float)
for row in np.arange(P_dimension):
    pt_array[row,0] = pt_tau_list[0][row]

# calculate step by step 
for i,t in enumerate(t_list[1:]) :
    t_old = t_list[i]
    p_old = pt_tau_list[i]
    if abs ( (t-t_old) - tau.evalf(2,subs=subs)) < 0.01 :
        newp = P_num*p_old
        pt_tau_list.append(newp)
        for row in np.arange(P_dimension):
            pt_array[row,i+1] = newp[row]
       
    else :
        print(f'the time list does not contain time step of tau')
        print(f' i = {i}, t_old ={t_old} , t = {t}')

#print(pt_tau_list)


**QUESTIONS**

1. Play with the initial concentrations and values of k to observe two exponential decays with different timescales. Try to associate a time-scale to a given exchange by looking at the time evolution.
2. Modify the pictures to write down on the picture the label of the numerical values.
3. Save the pictures with reasonable names.

In [ ]:


# plot probabilities as a function of time

exprA_eq_num = exprA_eq.evalf(2,subs=subs)
exprB_eq_num = exprB_eq.evalf(2,subs=subs)
exprC_eq_num = exprC_eq.evalf(2,subs=subs)

# define as much colors as dimension 
cm_subsection = np.linspace(0, 1, P_dimension) 
colors = [ cm.rainbow(x) for x in cm_subsection ]

fig, ax = plt.subplots()
for row in np.arange(P_dimension):
        ax.plot(t_list, pt_array[row,:],'o',color=colors[row],markersize=2, label = f'p{alphabet[row]} itterative')

ax.axhline(y=exprA_eq_num,label=f'pA(eq) = {exprA_eq_num}',linestyle=":",color=colors[0])
ax.axhline(y=exprB_eq_num,label=f'pB(eq) = {exprB_eq_num}',linestyle=":",color=colors[1])
ax.axhline(y=exprC_eq_num,label=f'pC(eq) = {exprC_eq_num}',linestyle=":",color=colors[2])

mytext =''
for val in subs :
        mytext = mytext + f' {val} {subs[val]}\n'
plt.text(t_list[-1]*0.8,0.1,mytext)

ax.set_xlabel('time')
ax.set_ylabel('probability')
plt.legend(loc='upper center')
plt.savefig(f'myfig.svg')
plt.show()


### Eigenvectors

In the following cell, einvectors are plotted. For each state, the value of the coefficient is plotted.

**QUESTIONS**

1. Add a line to put the associated time  with their associated timescales in the title 
2. Compare to the time evolution of concentration to recognize the associated transition. 

In [ ]:

def time_eigenvalue(l,tau):
    return -tau/(log(l))

# Plot the eigenvectors of the propagator with eigenvalues different from 1


for row,e in enumerate(eigen) :
    eigenvalue, multiplicity, vector = e   
    if eigenvalue == 1.0 :
        print(f"Eigenvalues = 1  ! \n This eigenvector describes the equilibrium distribution ")
        vector0 = vector[0]
        print(f'eigenvector = {vector0}')

        sum = 0.0
        for value in vector0:
            sum=sum+value
        vector_normalized = vector0/sum
        exprA_eq = vector_normalized[0].simplify()
        exprB_eq = vector_normalized[1].simplify()
        exprC_eq = vector_normalized[2].simplify()

        print(f"equilibrium distribution :")
        print(f'pA_eq = {exprA_eq}')
        print(f'pB_eq = {exprB_eq}')
        print(f'pC_eq = {exprC_eq}')
    else :
        #print(f'vector = {vector}')
        fig, ax = plt.subplots(1,1,figsize=(3,3),sharex=True)

        for v in vector:
            #print(v.evalf(2,subs=subs))
            ax.step(x=np.arange(P_dimension),y=v.evalf(5,subs=subs),where='mid')

        # change the labels and xtics
        #locs, labels = plt.xticks()
        positions = list(np.arange(P_dimension))
        labels = [ alphabet[p] for p in positions ]
        plt.xticks(positions,labels)
        plt.axhline(y=0,color='k')
        plt.title(f'Eigenvalue {eigenvalue.evalf(5,subs=subs)} => time {time_eigenvalue(eigenvalue.evalf(5,subs=subs), tau.evalf(5,subs=subs))}')
        #plt.title(TOCOMPLETE_EIGENVALUE_TIME)
        #plt.save(TOCOMPLETE_REASONABLE_TITLE)

        plt.show()

What have learnt from the practical ?